# Bayesian Synthetic Likelihood with ProbPipe

This tutorial walks through a complete likelihood-free inference pipeline
using ProbPipe primitives. We will:

1. **Define a stochastic simulator** (the Ricker model for population dynamics)
2. **Set up a Bayesian inverse problem** with synthetic data and summary statistics
3. **Implement synthetic likelihood** using `EmpiricalDistribution` and moment-matching
4. **Accelerate inference with a GP emulator** using `GaussianRandomFunction` and `EmulatorMixin`

Each step composes a small number of ProbPipe primitives. The emphasis is on
showing how complex workflows emerge from simple building blocks.

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from probpipe import (
    Normal,
    Gamma,
    ProductDistribution,
    EmpiricalDistribution,
    MultivariateNormal,
    GaussianRandomFunction,
    EmulatorMixin,
)

key = jax.random.PRNGKey(42)

## 1. The Stochastic Model

The **Ricker model** is a classic discrete-time population dynamics model
widely used as a benchmark in likelihood-free inference. The latent
population evolves as

$$N_{t+1} = r \, N_t \, \exp\!\left(-\frac{N_t}{K} + e_t\right), \qquad e_t \sim \mathcal{N}(0, \sigma^2),$$

and noisy observations are drawn as

$$Y_t \sim \mathrm{Poisson}(\phi \, N_t).$$

We fix $K = 10$ (carrying capacity) and $\phi = 1$ (observation scaling),
and aim to calibrate $\log r$ and $\sigma$ from an observed time series.

In [ ]:
def ricker_simulate(key, log_r, sigma, K=10.0, phi=1.0, T=50, N0=1.0):
    """Simulate a Ricker model time series.

    Parameters
    ----------
    key : PRNGKey
    log_r : float
        Log growth rate.
    sigma : float
        Process noise standard deviation.
    K : float
        Carrying capacity.
    phi : float
        Observation scaling.
    T : int
        Number of time steps.
    N0 : float
        Initial population.

    Returns
    -------
    Array, shape (T,)
        Poisson-observed population counts.
    """
    r = jnp.exp(log_r)

    def step(N, key_t):
        k1, k2 = jax.random.split(key_t)
        e = sigma * jax.random.normal(k1)
        N_next = r * N * jnp.exp(-N / K + e)
        N_next = jnp.clip(N_next, 0.0, 1e6)  # numerical safeguard
        Y = jax.random.poisson(k2, phi * N_next)
        return N_next, Y

    keys = jax.random.split(key, T)
    _, Y = jax.lax.scan(step, N0, keys)
    return Y

### Exploring model behaviour

The growth rate $\log r$ controls the qualitative dynamics: low values
give stable equilibria, while high values produce chaotic oscillations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)

log_r_values = [2.0, 3.0, 3.8]
sigma_fixed = 0.3

for ax, lr in zip(axes, log_r_values):
    for i in range(5):
        key, subkey = jax.random.split(key)
        y = ricker_simulate(subkey, lr, sigma_fixed)
        ax.plot(y, alpha=0.6, lw=0.8)
    ax.set_title(f"log r = {lr}", fontsize=10)
    ax.set_xlabel("t")

axes[0].set_ylabel("Observed count $Y_t$")
plt.suptitle(f"Ricker model trajectories ($\\sigma = {sigma_fixed}$)", fontsize=11)
plt.tight_layout()
plt.show()

The interaction between $\log r$ and $\sigma$ shapes both the amplitude
and regularity of the dynamics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True, sharex=True)

lr_vals = [3.0, 3.8]
sig_vals = [0.1, 0.5]

for i, lr in enumerate(lr_vals):
    for j, sig in enumerate(sig_vals):
        ax = axes[i, j]
        for _ in range(5):
            key, subkey = jax.random.split(key)
            y = ricker_simulate(subkey, lr, sig)
            ax.plot(y, alpha=0.5, lw=0.7)
        ax.set_title(f"log r = {lr}, $\\sigma$ = {sig}", fontsize=9)
        if i == 1:
            ax.set_xlabel("t")
        if j == 0:
            ax.set_ylabel("$Y_t$")

plt.suptitle("Ricker model: varying growth rate and noise", fontsize=11)
plt.tight_layout()
plt.show()

## 2. The Inverse Problem

We now set up a calibration problem. We generate synthetic observed data
at known true parameters, then attempt to recover those parameters using
Bayesian inference.

Since the Ricker model has no tractable likelihood, we will work with
**summary statistics** of the observed time series rather than the
full data.

In [ ]:
# True parameters
TRUE_LOG_R = 3.8
TRUE_SIGMA = 0.3
TRUE_K = 10.0

# Generate observed data
key, subkey = jax.random.split(key)
y_obs = ricker_simulate(subkey, TRUE_LOG_R, TRUE_SIGMA, K=TRUE_K, T=50)

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(y_obs, 'k-o', markersize=3, lw=1)
ax.set(xlabel="t", ylabel="$Y_t$", title="Observed data")
plt.tight_layout()
plt.show()

### Prior

We define the prior as a `ProductDistribution` over $\log r$ and
$\sigma$. This gives us a joint distribution with a `.log_prob()` method
that accepts parameter dictionaries.

In [ ]:
prior = ProductDistribution(
    log_r=Normal(loc=3.0, scale=1.0),
    sigma=Gamma(concentration=2.0, rate=5.0),
)

# Sample from the prior
key, subkey = jax.random.split(key)
theta_sample = prior.sample(subkey)
print("Sample from prior:", theta_sample)
print("Log-prior density:", prior.log_prob(theta_sample))

### Summary statistics

We compress each simulated time series into a low-dimensional summary
vector: the mean, variance, and lag-1 autocorrelation.

In [ ]:
def summary_stats(y):
    """Compute summary statistics of a time series."""
    y = jnp.asarray(y, dtype=jnp.float32)
    mu = jnp.mean(y)
    var = jnp.var(y)
    y_c = y - mu
    denom = jnp.sum(y_c ** 2)
    # Lag-1 autocorrelation (safe against zero variance)
    autocorr = jnp.where(
        denom > 0,
        jnp.sum(y_c[:-1] * y_c[1:]) / denom,
        0.0,
    )
    return jnp.array([mu, var, autocorr])


s_obs = summary_stats(y_obs)
print(f"Observed summaries: mean={s_obs[0]:.2f}, var={s_obs[1]:.1f}, autocorr={s_obs[2]:.3f}")

### Prior predictive check

Sampling from the prior and simulating shows the range of behaviours
that the model can produce. The observed data (red dashed) should look
plausible under the prior.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 5), sharey=True, sharex=True)

for ax in axes.flat:
    key, k1, k2 = jax.random.split(key, 3)
    theta = prior.sample(k1)
    y_sim = ricker_simulate(k2, theta["log_r"], theta["sigma"], K=TRUE_K)
    ax.plot(y_sim, 'steelblue', alpha=0.7, lw=0.8)
    ax.plot(y_obs, 'r--', alpha=0.4, lw=0.8)
    ax.set_title(
        f"lr={float(theta['log_r']):.2f}, $\\sigma$={float(theta['sigma']):.2f}",
        fontsize=8,
    )

axes[1, 0].set_xlabel("t")
axes[0, 0].set_ylabel("$Y_t$")
plt.suptitle("Prior predictive samples (red = observed)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of prior predictive summaries
n_prior = 300
key, subkey = jax.random.split(key)
prior_samples = prior.sample(subkey, sample_shape=(n_prior,))

prior_summaries = []
for i in range(n_prior):
    key, subkey = jax.random.split(key)
    y_sim = ricker_simulate(subkey, prior_samples["log_r"][i], prior_samples["sigma"][i], K=TRUE_K)
    prior_summaries.append(summary_stats(y_sim))
prior_summaries = jnp.stack(prior_summaries)

labels = ["Mean", "Variance", "Autocorrelation"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
pairs = [(0, 1), (0, 2), (1, 2)]

for ax, (i, j) in zip(axes, pairs):
    ax.scatter(prior_summaries[:, i], prior_summaries[:, j], alpha=0.3, s=8, c='steelblue')
    ax.plot(s_obs[i], s_obs[j], 'r*', markersize=12, zorder=5, label='observed')
    ax.set_xlabel(labels[i], fontsize=9)
    ax.set_ylabel(labels[j], fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle("Prior predictive summaries", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Synthetic Likelihood

The **synthetic likelihood** (Wood, 2010) approximates the intractable
likelihood by:

1. Simulating $N$ datasets at each candidate parameter $\theta$
2. Computing summary statistics for each simulation
3. Fitting a Gaussian to the empirical distribution of summaries
4. Evaluating the Gaussian density at the observed summaries

In ProbPipe, this composes naturally from three primitives:
`EmpiricalDistribution`, `MultivariateNormal.from_distribution()`
(moment-matching with provenance tracking), and `.log_prob()`.

In [ ]:
def synthetic_log_likelihood(key, theta, s_obs, N=200):
    """Evaluate the synthetic log-likelihood at parameter theta.

    Parameters
    ----------
    key : PRNGKey
    theta : dict
        Parameter dictionary with keys 'log_r' and 'sigma'.
    s_obs : Array, shape (3,)
        Observed summary statistics.
    N : int
        Number of simulations for the Monte Carlo approximation.

    Returns
    -------
    float
        Log synthetic likelihood.
    """
    keys = jax.random.split(key, N)

    # Step 1: simulate N datasets and compute summaries
    def sim_and_summarise(k):
        y = ricker_simulate(k, theta["log_r"], theta["sigma"], K=TRUE_K)
        return summary_stats(y)

    summaries = jax.vmap(sim_and_summarise)(keys)  # (N, 3)

    # Step 2: wrap as an EmpiricalDistribution
    emp = EmpiricalDistribution(summaries)

    # Step 3: moment-match to a MultivariateNormal
    mvn = MultivariateNormal.from_distribution(emp)

    # Step 4: evaluate log-density at observed summaries
    return mvn.log_prob(s_obs)

### Provenance tracking

ProbPipe records the `Empirical → MultivariateNormal` conversion as an
approximate operation. The provenance chain is accessible via `.source`.

In [ ]:
# Demonstrate provenance
key, subkey = jax.random.split(key)
demo_keys = jax.random.split(subkey, 200)
demo_summaries = jax.vmap(
    lambda k: summary_stats(ricker_simulate(k, TRUE_LOG_R, TRUE_SIGMA, K=TRUE_K))
)(demo_keys)

emp = EmpiricalDistribution(demo_summaries, name="simulated_summaries")
mvn = MultivariateNormal.from_distribution(emp, name="synthetic_likelihood")

print(f"Empirical mean: {emp.mean()}")
print(f"MVN mean:       {mvn.mean()}")
print(f"\nProvenance:")
print(f"  operation: {mvn.source.operation}")
print(f"  parent:    {type(mvn.source.parents[0]).__name__} ('{mvn.source.parents[0].name}')")
print(f"\nLog-likelihood at observed summaries: {mvn.log_prob(s_obs):.2f}")

### Likelihood surface

We evaluate the synthetic log-likelihood over a grid of $(\log r, \sigma)$
values to visualise the surface. This is expensive: each grid point
requires $N = 200$ simulator evaluations.

In [ ]:
n_grid = 20
log_r_grid = jnp.linspace(2.5, 4.5, n_grid)
sigma_grid = jnp.linspace(0.05, 0.8, n_grid)

key, subkey = jax.random.split(key)
grid_keys = jax.random.split(subkey, n_grid * n_grid)

t0 = time.time()
log_lik_surface = jnp.zeros((n_grid, n_grid))
for i, lr in enumerate(log_r_grid):
    for j, sig in enumerate(sigma_grid):
        theta = {"log_r": lr, "sigma": sig}
        k = grid_keys[i * n_grid + j]
        log_lik_surface = log_lik_surface.at[i, j].set(
            synthetic_log_likelihood(k, theta, s_obs, N=100)
        )
grid_time = time.time() - t0
print(f"Grid evaluation: {grid_time:.1f}s ({n_grid**2} points, N=200 sims each)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

# Clip for visualisation
vmax = jnp.nanmax(log_lik_surface)
levels = jnp.linspace(vmax - 30, vmax, 20)

cf = ax.contourf(
    sigma_grid, log_r_grid, log_lik_surface,
    levels=levels, cmap='viridis', extend='min',
)
plt.colorbar(cf, ax=ax, label='log synthetic likelihood')
ax.plot(TRUE_SIGMA, TRUE_LOG_R, 'w+', markersize=12, mew=2, label='true')
ax.set(xlabel=r'$\sigma$', ylabel=r'$\log r$')
ax.legend(fontsize=9)
ax.set_title('Synthetic likelihood surface', fontsize=11)
plt.tight_layout()
plt.show()

### Metropolis--Hastings sampling

We implement a simple random-walk Metropolis--Hastings sampler that uses
`prior.log_prob()` for the prior density and the synthetic likelihood
for the data term. ProbPipe provides the statistical building blocks;
the MCMC loop itself is intentionally simple.

In [ ]:
def run_mcmc(
    key, prior, s_obs, log_lik_fn, n_iter=2000,
    proposal_scale=None, N_sim=200,
):
    """Simple random-walk Metropolis-Hastings."""
    if proposal_scale is None:
        proposal_scale = jnp.array([0.15, 0.03])

    key, init_key, lik_key = jax.random.split(key, 3)
    theta = prior.sample(init_key)
    log_lik = log_lik_fn(lik_key, theta, s_obs, N=N_sim)
    log_prior = prior.log_prob(theta)

    chain_lr, chain_sig = [], []
    accept_count = 0

    for i in range(n_iter):
        key, prop_key, lik_key, u_key = jax.random.split(key, 4)
        noise = jax.random.normal(prop_key, shape=(2,)) * proposal_scale

        theta_prop = {
            "log_r": theta["log_r"] + noise[0],
            "sigma": jnp.maximum(theta["sigma"] + noise[1], 1e-3),
        }

        log_lik_prop = log_lik_fn(lik_key, theta_prop, s_obs, N=N_sim)
        log_prior_prop = prior.log_prob(theta_prop)

        log_alpha = (log_lik_prop + log_prior_prop) - (log_lik + log_prior)
        if jnp.log(jax.random.uniform(u_key)) < log_alpha:
            theta = theta_prop
            log_lik = log_lik_prop
            log_prior = log_prior_prop
            accept_count += 1

        chain_lr.append(float(theta["log_r"]))
        chain_sig.append(float(theta["sigma"]))

    print(f"Acceptance rate: {accept_count / n_iter:.1%}")
    return jnp.array(chain_lr), jnp.array(chain_sig)

In [ ]:
key, mcmc_key = jax.random.split(key)
t0 = time.time()
chain_lr, chain_sig = run_mcmc(
    mcmc_key, prior, s_obs, synthetic_log_likelihood,
    n_iter=800, N_sim=100,
)
sl_mcmc_time = time.time() - t0
print(f"MCMC time: {sl_mcmc_time:.0f}s")

In [ ]:
burnin = 500

fig, axes = plt.subplots(2, 2, figsize=(12, 5))

# Trace plots
axes[0, 0].plot(chain_lr, 'k-', lw=0.3)
axes[0, 0].axhline(TRUE_LOG_R, color='red', ls='--', lw=1, label='true')
axes[0, 0].axvline(burnin, color='grey', ls=':', lw=0.8)
axes[0, 0].set(ylabel=r'$\log r$', title='Trace: log r')
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(chain_sig, 'k-', lw=0.3)
axes[0, 1].axhline(TRUE_SIGMA, color='red', ls='--', lw=1, label='true')
axes[0, 1].axvline(burnin, color='grey', ls=':', lw=0.8)
axes[0, 1].set(ylabel=r'$\sigma$', title='Trace: $\\sigma$')
axes[0, 1].legend(fontsize=8)

# Marginal histograms (post burn-in)
axes[1, 0].hist(chain_lr[burnin:], bins=30, density=True, alpha=0.7, color='steelblue')
axes[1, 0].axvline(TRUE_LOG_R, color='red', ls='--', lw=1.5)
axes[1, 0].set(xlabel=r'$\log r$', ylabel='density')

axes[1, 1].hist(chain_sig[burnin:], bins=30, density=True, alpha=0.7, color='steelblue')
axes[1, 1].axvline(TRUE_SIGMA, color='red', ls='--', lw=1.5)
axes[1, 1].set(xlabel=r'$\sigma$', ylabel='density')

plt.suptitle('Synthetic likelihood MCMC', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(chain_sig[burnin:], chain_lr[burnin:], alpha=0.3, s=5, c='steelblue', label='posterior')
ax.plot(TRUE_SIGMA, TRUE_LOG_R, 'r*', markersize=14, zorder=5, label='true')
ax.set(xlabel=r'$\sigma$', ylabel=r'$\log r$', title='Joint posterior (synthetic likelihood)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

The synthetic likelihood posterior concentrates around the true parameters.
However, this came at a significant computational cost: each of the
800 MCMC steps required 100 simulator runs.

Can we do better?

## 4. GP-Accelerated Synthetic Likelihood

Instead of re-simulating at every MCMC iteration, we build a **GP
emulator** of the log synthetic likelihood surface. Once trained on a
modest design set, the GP provides near-instant predictions with
calibrated uncertainty.

We define a simple GP using ProbPipe's `GaussianRandomFunction` and
`EmulatorMixin` interfaces. In practice, ProbPipe's converter
infrastructure makes it straightforward to wrap GP models from
dedicated packages such as GPJax.

In [ ]:
def rbf_kernel(X1, X2, lengthscale=1.0, variance=1.0):
    """Squared-exponential (RBF) kernel."""
    sq_dist = jnp.sum((X1[:, None, :] - X2[None, :, :]) ** 2, axis=-1)
    return variance * jnp.exp(-0.5 * sq_dist / lengthscale ** 2)


class GPEmulator(GaussianRandomFunction, EmulatorMixin):
    """Simple GP emulator with an RBF kernel."""

    supports_joint_inputs = True

    def __init__(self, lengthscale=1.0, variance=1.0, noise=0.01):
        super().__init__(input_shape=(2,), output_shape=())
        self._ls = lengthscale
        self._var = variance
        self._noise = noise
        self._X_train = None
        self._Y_train = None
        self._alpha = None
        self._L = None

    def fit(self, X, Y):
        self._X_train = jnp.asarray(X)
        self._Y_train = jnp.asarray(Y)
        K = rbf_kernel(self._X_train, self._X_train, self._ls, self._var)
        K += self._noise * jnp.eye(K.shape[0])
        self._L = jnp.linalg.cholesky(K)
        self._alpha = jax.scipy.linalg.cho_solve((self._L, True), self._Y_train)

    @property
    def training_inputs(self):
        return self._X_train

    @property
    def training_responses(self):
        return self._Y_train

    def predict_mean(self, X):
        extra_batch, n = self._parse_X(X)
        if self._X_train is None:
            return jnp.zeros((*extra_batch, n))
        X_flat = X.reshape(-1, 2)
        k_star = rbf_kernel(X_flat, self._X_train, self._ls, self._var)
        mu = (k_star @ self._alpha).reshape(*extra_batch, n)
        return mu

    def predict_variance(self, X):
        extra_batch, n = self._parse_X(X)
        if self._X_train is None:
            return jnp.full((*extra_batch, n), self._var)
        X_flat = X.reshape(-1, 2)
        k_star = rbf_kernel(X_flat, self._X_train, self._ls, self._var)
        v = jax.scipy.linalg.solve_triangular(self._L, k_star.T, lower=True)
        var = self._var - jnp.sum(v ** 2, axis=0)
        return jnp.maximum(var, 1e-10).reshape(*extra_batch, n)

    def predict_covariance(self, X, *, joint_inputs=False, joint_outputs=False):
        if not joint_inputs:
            raise NotImplementedError("Only joint_inputs=True is supported.")
        extra_batch, n = self._parse_X(X)
        X_flat = X.reshape(-1, 2)
        K_star = rbf_kernel(X_flat, X_flat, self._ls, self._var)
        if self._X_train is None:
            cov = K_star + self._noise * jnp.eye(n)
        else:
            k_cross = rbf_kernel(X_flat, self._X_train, self._ls, self._var)
            v = jax.scipy.linalg.solve_triangular(self._L, k_cross.T, lower=True)
            cov = K_star - v.T @ v + 1e-6 * jnp.eye(K_star.shape[0])
        return cov.reshape(*extra_batch, n, n)

### Training data

We generate a set of design points in the parameter space and evaluate
the (expensive) synthetic log-likelihood at each. This is a one-time
cost that replaces the repeated simulation at every MCMC step.

In [ ]:
n_design = 80
key, k1, k2 = jax.random.split(key, 3)

# Random design in the parameter region of interest
log_r_design = jax.random.uniform(k1, shape=(n_design,), minval=2.5, maxval=4.5)
sigma_design = jax.random.uniform(k2, shape=(n_design,), minval=0.05, maxval=0.8)
X_design = jnp.stack([log_r_design, sigma_design], axis=-1)  # (n_design, 2)

# Evaluate synthetic log-likelihood at each design point
key, subkey = jax.random.split(key)
eval_keys = jax.random.split(subkey, n_design)

t0 = time.time()
Y_design = jnp.array([
    float(synthetic_log_likelihood(
        eval_keys[i],
        {"log_r": X_design[i, 0], "sigma": X_design[i, 1]},
        s_obs, N=100,
    ))
    for i in range(n_design)
])
design_time = time.time() - t0
print(f"Design evaluation: {design_time:.1f}s ({n_design} points)")
print(f"Log-likelihood range: [{float(jnp.nanmin(Y_design)):.1f}, {float(jnp.nanmax(Y_design)):.1f}]")

In [ ]:
# Standardise targets for GP conditioning
finite_mask = jnp.isfinite(Y_design)
Y_mean = jnp.mean(Y_design, where=finite_mask)
Y_std = jnp.std(Y_design, where=finite_mask)
Y_std = jnp.where(Y_std > 0, Y_std, 1.0)

# Replace non-finite values with a low floor
Y_clean = jnp.where(finite_mask, Y_design, Y_mean - 3 * Y_std)
Y_standardised = (Y_clean - Y_mean) / Y_std

# Fit GP emulator
gp_emu = GPEmulator(lengthscale=0.5, variance=1.0, noise=0.05)
gp_emu.fit(X_design, Y_standardised)

print(f"GP trained on {gp_emu.training_inputs.shape[0]} points")
print(f"isinstance check: GaussianRandomFunction={isinstance(gp_emu, GaussianRandomFunction)}, "
      f"EmulatorMixin={isinstance(gp_emu, EmulatorMixin)}")

### GP fit quality

Comparing the GP predictive mean against the brute-force surface confirms
that the emulator captures the likelihood landscape. The predictive
standard deviation shows where the GP is most uncertain.

In [ ]:
# Predict on the same grid used for the brute-force surface
LR_grid, SIG_grid = jnp.meshgrid(log_r_grid, sigma_grid, indexing='ij')
X_grid = jnp.stack([LR_grid.ravel(), SIG_grid.ravel()], axis=-1)  # (n_grid^2, 2)

pred = gp_emu(X_grid)  # returns Normal distribution
gp_mean = (pred.mean() * Y_std + Y_mean).reshape(n_grid, n_grid)
gp_std = (jnp.sqrt(pred.variance()) * Y_std).reshape(n_grid, n_grid)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Brute-force surface
vmax = float(jnp.nanmax(log_lik_surface))
levels = jnp.linspace(vmax - 30, vmax, 20)
cf0 = axes[0].contourf(sigma_grid, log_r_grid, log_lik_surface, levels=levels, cmap='viridis', extend='min')
axes[0].set_title('Brute-force surface', fontsize=10)
plt.colorbar(cf0, ax=axes[0])

# GP mean
cf1 = axes[1].contourf(sigma_grid, log_r_grid, gp_mean, levels=levels, cmap='viridis', extend='min')
axes[1].scatter(X_design[:, 1], X_design[:, 0], c='white', s=6, alpha=0.5, edgecolors='none')
axes[1].set_title('GP predictive mean', fontsize=10)
plt.colorbar(cf1, ax=axes[1])

# GP std
cf2 = axes[2].contourf(sigma_grid, log_r_grid, gp_std, levels=15, cmap='magma')
axes[2].scatter(X_design[:, 1], X_design[:, 0], c='white', s=6, alpha=0.5, edgecolors='none')
axes[2].set_title('GP predictive std', fontsize=10)
plt.colorbar(cf2, ax=axes[2])

for ax in axes:
    ax.plot(TRUE_SIGMA, TRUE_LOG_R, 'w+', markersize=10, mew=2)
    ax.set(xlabel=r'$\sigma$', ylabel=r'$\log r$')

plt.suptitle('Synthetic likelihood: brute-force vs GP emulator', fontsize=11)
plt.tight_layout()
plt.show()

### GP-accelerated MCMC

We now run MCMC using the GP emulator in place of the simulator-based
synthetic likelihood. Each likelihood evaluation is now a cheap
matrix--vector product rather than 200 simulations.

In [ ]:
def gp_log_likelihood(key, theta, s_obs, N=None):
    """GP-emulated log-likelihood (key and N are unused, kept for API compat)."""
    x = jnp.array([[theta["log_r"], theta["sigma"]]])
    pred = gp_emu(x)
    return pred.mean()[0] * Y_std + Y_mean  # un-standardise


key, mcmc_key = jax.random.split(key)
t0 = time.time()
chain_gp_lr, chain_gp_sig = run_mcmc(
    mcmc_key, prior, s_obs, gp_log_likelihood,
    n_iter=3000, N_sim=100,
)
gp_mcmc_time = time.time() - t0
print(f"GP-MCMC time: {gp_mcmc_time:.1f}s")

In [ ]:
burnin_gp = 500

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Brute-force posterior
axes[0].scatter(
    chain_sig[burnin:], chain_lr[burnin:],
    alpha=0.3, s=5, c='steelblue', label='SL posterior',
)
axes[0].plot(TRUE_SIGMA, TRUE_LOG_R, 'r*', markersize=14, zorder=5, label='true')
axes[0].set(xlabel=r'$\sigma$', ylabel=r'$\log r$',
            title='Brute-force synthetic likelihood')
axes[0].legend(fontsize=8)

# GP-accelerated posterior
axes[1].scatter(
    chain_gp_sig[burnin_gp:], chain_gp_lr[burnin_gp:],
    alpha=0.3, s=5, c='orange', label='GP-SL posterior',
)
axes[1].plot(TRUE_SIGMA, TRUE_LOG_R, 'r*', markersize=14, zorder=5, label='true')
axes[1].set(xlabel=r'$\sigma$', ylabel=r'$\log r$',
            title='GP-accelerated synthetic likelihood')
axes[1].legend(fontsize=8)

# Match axis limits
for ax in axes:
    ax.set_xlim(0.0, 0.85)
    ax.set_ylim(2.5, 4.5)

plt.suptitle('Posterior comparison', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
print("Computational cost comparison")
print("=" * 45)
print(f"Brute-force SL MCMC (800 iters): {sl_mcmc_time:8.1f}s")
print(f"GP design evaluation ({n_design} points):  {design_time:8.1f}s")
print(f"GP-accelerated MCMC (3000 iters):  {gp_mcmc_time:8.1f}s")
print(f"GP total (design + MCMC):           {design_time + gp_mcmc_time:8.1f}s")
print(f"\nSpeedup per MCMC step: ~{sl_mcmc_time / 800 / (gp_mcmc_time / 3000):.0f}x")

## Summary

This tutorial demonstrated a full likelihood-free inference workflow
built from a small number of ProbPipe primitives:

| Primitive | Role |
|:--|:--|
| `ProductDistribution` | Composed the joint prior with `log_prob()` for MCMC |
| `EmpiricalDistribution` | Wrapped simulation outputs with statistical semantics |
| `MultivariateNormal.from_distribution()` | Moment-matched to a Gaussian with provenance |
| `GaussianRandomFunction` + `EmulatorMixin` | Drop-in GP surrogate for the likelihood |

**Extensions.** Rather than emulating the log-likelihood directly,
one can emulate each summary statistic as a separate
`GaussianRandomFunction` and then construct the synthetic likelihood
analytically from the GP predictive distributions. This is more natural
when reasoning about individual statistics but requires multi-output GP
modelling.

**Note on the GP.** The `GPEmulator` defined above uses fixed
hyperparameters for simplicity. In practice, one would optimise the
kernel hyperparameters via marginal likelihood maximisation. ProbPipe's
converter infrastructure makes it straightforward to wrap GP models
from dedicated packages such as GPJax.